In [1]:
import numpy as np

def freq_to_word(f):
    # f in Hz
    if f < 0 or f >= 1e9:
        logging.warning("freq needs to be in range [0,1e9)")
        num = 0
    num = round(2**32/1e9*f) & 0xffff_ffff
    return (f"{num:0{8}x}")

def dec2hex(dec_num):
    return hex(int(np.round(dec_num,0)))
    
def hex2dec(hex_num):
    # convert a hex number to a decimal
    bits_num = 16
    return int(hex_num, bits_num)

In [19]:
def set_fm(channel, f_min, f_max):

    # Find minimum and maximum frequency tuning word, i.e. min/max frequencies of the modulation 
    min_freqtuningword = freq_to_word(f_min)    
    max_freqtuningword = freq_to_word(f_max)      

    print(min_freqtuningword)
    print(max_freqtuningword)

    # min ans max of D values (modulation data, see manual)
    d_min = hex2dec(min_freqtuningword)/2**12
    d_max = hex2dec(max_freqtuningword)/2**12


    print(d_min)
    print(d_max)

    print()
    # find scaling factor S0 and offset O:
    print((d_max - d_min)*2**12 / (2**15 - 1 + 2**15))
    S0 = str(dec2hex((d_max - d_min)*2**12 / (2**15 - 1 + 2**15)))
    O = str(dec2hex((d_min*(2**15 - 1) - d_max*(-2**15)) / (2**15 - 1 + 2**15)))


    print()
    # choose max amplitude and set FTW to zero
    print(f'dcp {channel} spi:STP0=0x3fff000000000000')

    # enable parallel data port, set FM gain to 12 -> look at AD9910 datasheet for FM gain, I have not understood it.
    print(f'dcp {channel} spi:CFR2=0b00000000_00000000_01011100')
    
    # set scale factor S0
    print(f'dcp {channel} wr:AM_S0={S0}')
    # set channel offset O0: here O0 = 0 
    print(f'dcp {channel} wr:AM_O0=0') 
    # set global offset O:
    print(f'dcp {channel} wr:AM_O={O}') 
    #print(f'dcp {channel} wr:AM_O=0x51eb') 
    # choose frequency modulation, flush coeff.
    print(f'dcp {channel} wr:AM_CFG=0x2000_0002') 
    return

set_fm(0, 50e6, 70e6)

0ccccccd
11eb851f
52428.800048828125
73400.32006835938

1310.7400015259022

dcp 0 spi:STP0=0x3fff000000000000
dcp 0 spi:CFR2=0b00000000_00000000_01011100
dcp 0 wr:AM_S0=0x51f
dcp 0 wr:AM_O0=0
dcp 0 wr:AM_O=0xf5c3
dcp 0 wr:AM_CFG=0x2000_0002


In [7]:
set_fm(0, 10e6, 30e6)

028f5c29
07ae147b
0x51ebd7
dcp 0 spi:STP0=0x3fff000000000000
dcp 0 spi:CFR1=0b01000001_00000000_00000000
dcp 0 spi:CFR2=0b00000000_00000000_01011100
dcp 0 wr:AM_S0=0x51ebd7
dcp 0 wr:AM_O0=0
dcp 0 wr:AM_O=0x51ebae1
dcp 0 wr:AM_CFG=0x2000_0002
